# Everpure CLV Predictive Analytics
## 01 — Dataset Understanding

Objective:
Understand the dataset structure, business variables,
and potential prediction targets before preprocessing.

Dataset: Case Study 2 — CLV Predictive Analytics

In [1]:
import pandas as pd
from pathlib import Path

# Path to the original dataset
DATA_PATH = Path("../data/raw/Case_Study_2_CLV_Predictive_Analytics.csv")

# Load CSV into a pandas DataFrame
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [2]:
# Dataset dimensions
print("Dataset shape:", df.shape)

# Number of records
print("Total rows:", df.shape[0])

# Number of variables
print("Total columns:", df.shape[1])

# Display column names
print("\nColumn names:")
for col in df.columns:
    print("-", col)

Dataset shape: (2000, 9)
Total rows: 2000
Total columns: 9

Column names:
- account_id
- array_id
- contract_end_date
- historical_15yr_net_profit_usd
- current_capacity_tb
- utilization_rate_pct
- static_baseline_churn_rate
- churn_event_next_12m
- expansion_potential_signal


In [3]:
# Display the first 5 records
df.head()

,account_id,array_id,contract_end_date,historical_15yr_net_profit_usd,current_capacity_tb,utilization_rate_pct,static_baseline_churn_rate,churn_event_next_12m,expansion_potential_signal
0,ACC_1000,ARR_5000,2026-10-01,146797,1382,0.59,0.05,0.0,Medium
1,ACC_1001,ARR_5001,2026-10-01,1236408,1809,0.93,0.05,1.0,UNKNOWN
2,ACC_1002,ARR_5002,2026-10-02,2790632,605,1.28,0.05,0.0,Low
3,ACC_1003,ARR_5003,2026-10-02,1233288,1567,0.47,0.05,0.0,Low
4,ACC_1004,ARR_5004,2026-10-03,2881167,488,1.22,0.05,0.0,Low


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   account_id                      2000 non-null   str    
 1   array_id                        2000 non-null   str    
 2   contract_end_date               2000 non-null   str    
 3   historical_15yr_net_profit_usd  2000 non-null   int64  
 4   current_capacity_tb             2000 non-null   int64  
 5   utilization_rate_pct            2000 non-null   float64
 6   static_baseline_churn_rate      2000 non-null   float64
 7   churn_event_next_12m            1884 non-null   float64
 8   expansion_potential_signal      2000 non-null   str    
dtypes: float64(3), int64(2), str(4)
memory usage: 200.1 KB


In [6]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

display(missing_summary)

,Missing Count,Missing %
account_id,0,0.0
array_id,0,0.0
contract_end_date,0,0.0
historical_15yr_net_profit_usd,0,0.0
current_capacity_tb,0,0.0
utilization_rate_pct,0,0.0
static_baseline_churn_rate,0,0.0
churn_event_next_12m,116,5.8
expansion_potential_signal,0,0.0


In [7]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

account_id: 1000 unique values
array_id: 2000 unique values
contract_end_date: 1000 unique values
historical_15yr_net_profit_usd: 2000 unique values
current_capacity_tb: 1366 unique values
utilization_rate_pct: 111 unique values
static_baseline_churn_rate: 1 unique values
churn_event_next_12m: 2 unique values
expansion_potential_signal: 4 unique values


In [8]:
print("Churn label distribution:")
print(df["churn_event_next_12m"].value_counts(dropna=False))

print("\nExpansion signal distribution:")
print(df["expansion_potential_signal"].value_counts(dropna=False))

Churn label distribution:
churn_event_next_12m
0.0    1586
1.0     298
NaN     116
Name: count, dtype: int64

Expansion signal distribution:
expansion_potential_signal
Low        841
Medium     622
High       426
UNKNOWN    111
Name: count, dtype: int64


### Prediction Target and Features

Target:
- churn_event_next_12m

Potential predictive features:
- contract_end_date
- historical_15yr_net_profit_usd
- current_capacity_tb
- utilization_rate_pct
- expansion_potential_signal

Identifiers:
- account_id
- array_id

Existing baseline:
- static_baseline_churn_rate

Important:
Feature availability and leakage risks must be checked before modeling.

In [9]:
print("Total records:", len(df))

print("Unique accounts:", df["account_id"].nunique())

print("Unique arrays:", df["array_id"].nunique())

print("\nArrays per account:")
print(df.groupby("account_id")["array_id"].nunique().describe())

print("\nExample accounts:")
display(
    df.groupby("account_id")["array_id"]
      .nunique()
      .head(10)
)

Total records: 2000
Unique accounts: 1000
Unique arrays: 2000

Arrays per account:
count    1000.0
mean        2.0
std         0.0
min         2.0
25%         2.0
50%         2.0
75%         2.0
max         2.0
Name: array_id, dtype: float64

Example accounts:


account_id
ACC_1000    2
ACC_1001    2
ACC_1002    2
ACC_1003    2
ACC_1004    2
ACC_1005    2
ACC_1006    2
ACC_1007    2
ACC_1008    2
ACC_1009    2
Name: array_id, dtype: int64

In [10]:
account_churn = (
    df.groupby("account_id")["churn_event_next_12m"]
      .agg(
          labeled_rows="count",
          distinct_labels=lambda x: x.dropna().nunique()
      )
)

print("Accounts with conflicting observed churn labels:")
print((account_churn["distinct_labels"] > 1).sum())

display(account_churn.head(10))

Accounts with conflicting observed churn labels:
224


,labeled_rows,distinct_labels
account_id,,
ACC_1000,2,2
ACC_1001,2,2
ACC_1002,2,2
ACC_1003,2,1
ACC_1004,1,1
ACC_1005,2,1
ACC_1006,1,1
ACC_1007,2,1
ACC_1008,2,1


In [11]:
feature_roles = pd.DataFrame({
    "Column": df.columns,
    "Role": [
        "Account identifier",
        "Array identifier",
        "Potential feature - date",
        "Potential feature - leakage review",
        "Potential feature - capacity",
        "Potential feature - utilization",
        "Baseline only",
        "Target",
        "Potential feature - categorical"
    ]
})

display(feature_roles)

,Column,Role
0,account_id,Account identifier
1,array_id,Array identifier
2,contract_end_date,Potential feature - date
3,historical_15yr_net_profit_usd,Potential feature - leakage review
4,current_capacity_tb,Potential feature - capacity
5,utilization_rate_pct,Potential feature - utilization
6,static_baseline_churn_rate,Baseline only
7,churn_event_next_12m,Target
8,expansion_potential_signal,Potential feature - categorical


## Initial Findings — Account and Array Structure

1. The dataset contains 2,000 storage-array records associated
   with 1,000 unique customer accounts.

2. Every account has exactly two arrays.

3. There are 224 accounts with differing observed churn labels
   across their arrays.

4. This distinction suggests that an array churn event cannot
   automatically be interpreted as complete customer churn.

5. To reduce data leakage, all arrays belonging to the same
   account will remain in the same training, validation, or
   test partition.

6. The primary prediction target is array-level churn within
   the next 12 months.

7. The definition of account-level churn requires additional
   business information about customer relationships and
   array retirement events.

In [12]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isna().sum())

print("\nTARGET DISTRIBUTION")
print(df["churn_event_next_12m"].value_counts(dropna=False))

print("\nEXPANSION SIGNAL DISTRIBUTION")
print(df["expansion_potential_signal"].value_counts(dropna=False))

DATA TYPES
account_id                            str
array_id                              str
contract_end_date                     str
historical_15yr_net_profit_usd      int64
current_capacity_tb                 int64
utilization_rate_pct              float64
static_baseline_churn_rate        float64
churn_event_next_12m              float64
expansion_potential_signal            str
dtype: object

MISSING VALUES
account_id                          0
array_id                            0
contract_end_date                   0
historical_15yr_net_profit_usd      0
current_capacity_tb                 0
utilization_rate_pct                0
static_baseline_churn_rate          0
churn_event_next_12m              116
expansion_potential_signal          0
dtype: int64

TARGET DISTRIBUTION
churn_event_next_12m
0.0    1586
1.0     298
NaN     116
Name: count, dtype: int64

EXPANSION SIGNAL DISTRIBUTION
expansion_potential_signal
Low        841
Medium     622
High       426
UNKNOWN    111
Name